In [23]:
# Import necessary libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import pickle
from datetime import datetime
import statsmodels.api as sm
from sklearn.cluster import KMeans
import warnings

warnings.filterwarnings('ignore')

# Import our MarkovKAMA implementation
from markovKama import MarkovKAMA

In [24]:
# Path to Bitcoin data
data_path = 'data/micro/candleData/bitcoin_candles.csv'

# Initialize the MarkovKAMA model
model = MarkovKAMA(data_path=data_path, train_test_split=0.6, init_date='2013-01-01',embargo_pct=0.005)

# Display the first few rows of the data
model.data.head()

Data loaded successfully: 4350 records
Training set: 2610 records
Test set: 1740 records


,open,high,low,close
date,,,,
2013-04-28,135.30,135.30,135.30,135.30
2013-04-29,141.96,141.96,141.96,141.96
2013-04-30,135.30,135.30,135.30,135.30
2013-05-01,117.00,117.00,117.00,117.00
2013-05-02,103.43,103.43,103.43,103.43


In [25]:
# Apply the Markov Switching Regression model
low_var, high_var = model.get_markov(no_regimes=2, return_data=True)

# Plot the probabilities of each regime over time using plotly as a stacked area chart
fig = go.Figure()

# Create a stacked area chart since low_var + high_var = 1
fig.add_trace(
    go.Scatter(
        x=low_var.index, 
        y=low_var, 
        name='Low Volatility',
        line=dict(width=0),
        stackgroup='one',
        fillcolor='rgba(0, 255, 0, 0.7)'
    )
)

fig.add_trace(
    go.Scatter(
        x=high_var.index, 
        y=high_var, 
        name='High Volatility',
        line=dict(width=0),
        stackgroup='one',
        fillcolor='rgba(255, 0, 0, 0.7)'
    )
)

# Calculate the test data start date
test_start_idx = int(len(model.data) * model.train_test_split)
test_start_date = model.data.index[test_start_idx]

# Add vertical line for test data start - using a scatter trace instead of add_vline
fig.add_trace(
    go.Scatter(
        x=[test_start_date, test_start_date],
        y=[0, 1],
        mode='lines',
        line=dict(color='black', width=2, dash='dash'),
        name='Test Data Start',
        showlegend=True
    )
)

# Add annotation for the test data line
fig.add_annotation(
    x=test_start_date,
    y=1,
    text="",
    showarrow=False,
    yshift=10
)

# Update layout
fig.update_layout(
    title='Markov Switching Regime Probabilities for Bitcoin',
    xaxis_title='Date',
    yaxis_title='Probability',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    height=600,
    width=1000,
    hovermode='x unified'
)

fig.show()

Markov Switching Regression model fitted successfully.
Training data: 2589 points
Embargo data: 21 points
Test data: 1738 points


In [26]:
# Calculate KAMA with default parameters
# There seems to be an issue with the model.get_kama method
# Let's fix the approach by using the correct method call

# First, check if kama is already calculated and stored as an attribute
if hasattr(model, 'kama_values') and isinstance(model.kama_values, pd.Series):
    kama_ = model.kama_values
else:
    # If not, call the method without trying to unpack multiple return values
    model.get_kama(n_window=10, pow1=2, pow2=30, gamma=0.15)
    kama_ = model.kama_values

# Plot BTC price with KAMA using Plotly
# Note: KAMA values are in log scale, so we need to transform them back
price = model.asset_data
# Convert log KAMA values to actual price scale
kama_actual = np.exp(kama_)

fig = go.Figure()

# Add price trace
fig.add_trace(
    go.Scatter(
        x=price.index,
        y=price,
        name='BTC Price',
        line=dict(color='blue', width=1),
        opacity=0.5
    )
)

# Add KAMA trace (converted from log scale)
fig.add_trace(
    go.Scatter(
        x=kama_actual.index,
        y=kama_actual,
        name='KAMA',
        line=dict(color='black', width=2)
    )
)

# Update layout
fig.update_layout(
    title='Bitcoin Price and KAMA',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    height=600,
    width=1000,
    hovermode='x unified'
)

fig.show()

KAMA calculation completed successfully.


In [27]:
# Classify market regimes
classes = model.get_classes()

# Count occurrences of each regime
regime_counts = classes['label'].value_counts()
print("Regime Counts:")
print(regime_counts)

# Calculate metrics for each regime segment
metrics = model.calculate_metrics(classes)

# Display regime metrics
print("\nRegime Metrics:")
display(metrics[['label', 'start_date', 'end_date', 'duration', 'slope', 'volatility', 'returns']].tail(10))

Regime classification completed successfully.
Regime Counts:
label
Bullish_Low_Var     2460
Bearish_Low_Var      852
Bearish_High_Var     534
Bullish_High_Var     501
Name: count, dtype: int64

Regime Metrics:


,label,start_date,end_date,duration,slope,volatility,returns
398,Bullish_Low_Var,2024-11-11,2024-12-27,47,0.001889,0.024196,0.078400
399,Bearish_Low_Var,2024-12-28,2025-01-03,7,0.010474,0.014406,0.049048
400,Bullish_Low_Var,2025-01-04,2025-02-06,34,0.001225,0.023972,-0.017588
401,Bearish_Low_Var,2025-02-07,2025-02-16,10,0.000478,0.012103,-0.004236
402,Bullish_Low_Var,2025-02-17,2025-02-24,8,-0.003656,0.022351,-0.045721
403,Bearish_High_Var,2025-02-25,2025-02-27,3,-0.023335,0.033112,-0.045597
404,Bearish_High_Var,2025-03-01,2025-03-10,10,-0.011493,0.051839,-0.083960
405,Bullish_Low_Var,2025-03-11,2025-03-12,2,0.013019,0.025942,0.013104
406,Bearish_Low_Var,2025-03-13,2025-03-21,9,0.003506,0.028593,0.035894
407,Bullish_Low_Var,2025-03-22,2025-03-25,4,0.014838,0.012419,0.044491


In [28]:
print(metrics.groupby('label')['returns'].mean())
print(metrics.groupby('label')['returns'].std())
print(metrics.groupby('label')['returns'].median())
print(metrics.groupby('label')['returns'].skew())



label
Bearish_High_Var   -0.003098
Bearish_Low_Var     0.008839
Bullish_High_Var    0.069813
Bullish_Low_Var     0.032101
Name: returns, dtype: float64
label
Bearish_High_Var    0.133881
Bearish_Low_Var     0.041413
Bullish_High_Var    0.338361
Bullish_Low_Var     0.086715
Name: returns, dtype: float64
label
Bearish_High_Var   -0.010997
Bearish_Low_Var     0.008458
Bullish_High_Var    0.002564
Bullish_Low_Var     0.010982
Name: returns, dtype: float64
label
Bearish_High_Var    0.015867
Bearish_Low_Var    -0.059335
Bullish_High_Var    5.915563
Bullish_Low_Var     1.103403
Name: returns, dtype: float64


In [29]:
portfolio = model.implement_trading_strategy(classes, initial_capital=1000)
portfolio

Regime exposure (average position & % of time):
  Bullish_Low_Var: 0.92 (62.3% of time)
  Bullish_High_Var: 0.33 (8.4% of time)
  Bearish_Low_Var: 0.06 (19.7% of time)
  Bearish_High_Var: 0.10 (9.6% of time)


,btc_price,regime,next_open,trading_state,cash,btc_holdings,portfolio_value,trades,trading_costs,cumulative_costs,strategy_returns,btc_returns,buy_hold_value,strategy_normalized,buy_hold_normalized,regime_change,regime_duration
date,,,,,,,,,,,,,,,,,
2020-06-22,9678.68,Bullish_Low_Var,9678.71,0,1000.000000,0.000000,1000.000000,0,0,0,NaN,NaN,1000.000000,1.000000,1.000000,True,13
2020-06-23,9624.68,Bullish_Low_Var,9624.24,1,0.000000,0.103216,993.423227,1,1,1,-0.006577,-0.005579,994.420727,0.993423,0.994421,False,13
2020-06-24,9288.06,Bullish_Low_Var,9290.68,1,0.000000,0.103216,958.678578,0,0,1,-0.034975,-0.034975,959.641191,0.958679,0.959641,False,13
2020-06-25,9258.67,Bullish_Low_Var,9258.25,1,0.000000,0.103216,955.645053,0,0,1,-0.003164,-0.003164,956.604620,0.955645,0.956605,False,13
2020-06-26,9166.49,Bullish_Low_Var,9150.77,1,0.000000,0.103216,946.130580,0,0,1,-0.009956,-0.009956,947.080594,0.946131,0.947081,False,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-21,84010.00,Bearish_Low_Var,84046.00,0,2333.715822,0.000000,2333.715822,0,0,170,0.000000,-0.003097,8679.902631,2.333716,8.679903,False,9
2025-03-22,83793.00,Bullish_Low_Var,83823.00,0,2333.715822,0.000000,2333.715822,0,0,170,0.000000,-0.002583,8657.482219,2.333716,8.657482,True,4
2025-03-23,85788.00,Bullish_Low_Var,86114.00,1,0.000000,0.027829,2387.399937,1,1,171,0.023004,0.023809,8863.605368,2.387400,8.863605,False,4


In [30]:
# Generate visualization showing ONLY test set results
fig = model.plot_test_results(portfolio, classes)

# For deployment - using the latest data point's classifi

In [9]:
model.kama_values

date
2015-01-01     5.749361
2015-01-02     5.750825
2015-01-03     5.657983
2015-01-04     5.564290
2015-01-05     5.610277
                ...    
2025-03-15    11.384917
2025-03-16    11.381227
2025-03-17    11.380471
2025-03-18    11.379197
2025-03-19    11.378790
Name: close, Length: 3731, dtype: float64

In [10]:
model.optimize()

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

KAMA calculation completed successfully.              
Regime classification completed successfully.         
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        

{'gamma': 0.1, 'n': 20.0, 'pow1': 9.0, 'pow2': 80.0}

In [11]:
model.get_kama(n_window=15, pow1=5, pow2=55, gamma=0.15)

KAMA calculation completed successfully.
